# Graph-JEPA: `n local + m stratified random`

Сравнение downstream-метрик при фиксированных четырёх target-масках. По оси X — число stratified-random targets; число local targets равно `4 - m`. Baseline `2L+2R` включён в общую шкалу.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# При запуске локально укажи скачанный CSV; на сервере оставь абсолютный путь.
REPORT_CSV = Path('/workspace-SR004.nfs2/konovalov/tactile/experiments/significance/jepa_local_random_mix/significance_results.csv')
if not REPORT_CSV.exists():
    REPORT_CSV = Path('../experiments/significance/jepa_local_random_mix/significance_results.csv')

df = pd.read_csv(REPORT_CSV)
df = df[df['comparison_block_id'].eq('local_random_mix')].copy()

METHODS = {
    'graph_physical_dijkstra_1c4t_c50_90_t10_18': (0, '4 local'),
    'graph_physical_dijkstra_1c4t_3local1stratifiedrandom_c50_90_t10_18': (1, '3 local + 1 random'),
    'graph_physical_dijkstra_1c4t_2local2stratifiedrandom_c50_90_t10_18': (2, '2 local + 2 random'),
    'graph_physical_dijkstra_1c4t_1local3stratifiedrandom_c50_90_t10_18': (3, '1 local + 3 random'),
    'graph_physical_dijkstra_1c4t_4stratifiedrandom_c50_90_t10_18': (4, '4 random'),
}
df = df[df['experiment_id'].isin(METHODS)].copy()
df['random_targets'] = df['experiment_id'].map(lambda x: METHODS[x][0])
df['method_label'] = df['experiment_id'].map(lambda x: METHODS[x][1])
df = df.sort_values(['random_targets', 'is_baseline'])

print(f'Loaded {len(df)} metric rows from {REPORT_CSV}')
display(df[['task', 'metric', 'random_targets', 'method_label', 'estimate', 'bootstrap_se']].head())

In [ ]:
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid_alpha': 0.25})
COLORS = ['#2166ac', '#67a9cf', '#fdae61', '#d6604d', '#b2182b']

def series(task, metric):
    part = df[(df.task == task) & (df.metric == metric)].sort_values('random_targets')
    return part

def draw(ax, task, metric, title, ylabel, lower_is_better=False):
    part = series(task, metric)
    for i, (_, row) in enumerate(part.iterrows()):
        ax.errorbar(row.random_targets, row.estimate, yerr=row.bootstrap_se, marker='o',
                    color=COLORS[int(row.random_targets)], capsize=3,
                    label=row.method_label if i == 0 else None)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(5))
    ax.set_xlabel('Число stratified-random targets (m)')
    if lower_is_better:
        ax.text(0.98, 0.04, '↓ лучше', transform=ax.transAxes, ha='right', va='bottom', fontsize=9)
    else:
        ax.text(0.98, 0.04, '↑ лучше', transform=ax.transAxes, ha='right', va='bottom', fontsize=9)

# 1. Force: общий RMSE и RMSE по координатам.
fig = plt.figure(figsize=(16, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 3)
ax = fig.add_subplot(gs[0, :])
draw(ax, 'force', 'rmse', 'Force: общий RMSE', 'RMSE', lower_is_better=True)
for j, coord in enumerate(['x', 'y', 'z']):
    draw(fig.add_subplot(gs[1, j]), 'force', f'rmse_{coord}', f'Force: RMSE {coord}', 'RMSE', lower_is_better=True)
fig.suptitle('Force downstream', fontsize=16)
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 0.98))
plt.show()

In [ ]:
# 2. Pose estimation: для каждой координаты показываем RMSE и accuracy.
fig, axes = plt.subplots(3, 2, figsize=(15, 13), constrained_layout=True)
for i, coord in enumerate(['x', 'y', 'theta']):
    draw(axes[i, 0], 'pose', f'rmse_{coord}', f'Pose: RMSE {coord}', 'RMSE', lower_is_better=True)
    draw(axes[i, 1], 'pose', f'acc_{coord}', f'Pose: accuracy {coord}', 'Accuracy', lower_is_better=False)
fig.suptitle('Pose estimation downstream', fontsize=16)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 0.98))
plt.show()

In [ ]:
# 3. Object classification.
fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
draw(ax, 'object_classification', 'acc', 'Object classification: accuracy', 'Accuracy', lower_is_better=False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, ncol=5)
plt.show()